# Here, we will fine-tune a LLAMA model to prevent overfiltering

Here, we import all the libraries that we need to run the code

In [ ]:
import json
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

## Now, we load the dataset to fine-tune model on it

In [ ]:
with open("train.json") as f:
    raw_data = json.load(f)

records = [{"text": d["prompt"], "label": int(d["label"])} for d in raw_data]

dataset = Dataset.from_list(records)
dataset = dataset.class_encode_column("label")
split = dataset.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds, eval_ds = split["train"], split["test"]

print(f"Train size: {len(train_ds)}  |  Eval size: {len(eval_ds)}")